In [7]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv
from datetime import datetime

# Carregar variáveis de ambiente
load_dotenv("/root/projeto-conti-rh/conti-rh-backend/.env")

# Inicializar o cliente MongoDB manualmente
class Mongo:
    def __init__(self, uri):
        self.client = MongoClient(uri)
        self.db = self.client.get_database()

# Instanciar a conexão com o banco
mongo = Mongo(os.getenv("MONGO_URI"))


In [8]:
def calcular_folha_pagamento_fixo(data_inicial: datetime, data_final: datetime):
    """
    Calcula a folha de pagamento com base na proporção de dias corridos segundo a CLT (mês comercial de 30 dias),
    levando em consideração a data de admissão do colaborador.

    Parâmetros:
    - data_inicial (datetime): Data inicial do período.
    - data_final (datetime): Data final do período.

    Retorno:
    - list: Lista de colaboradores com o valor de pagamento proporcional, dias corridos e horas proporcionais.
    """
    db = mongo.db  # Acessa o banco de dados usando a instância global
    dias_no_mes = 30

    pipeline = [
        {"$match": {"ativo": True, "dados_contrato.tipo_remuneracao": "Fixo"}},
        {
            "$addFields": {
                "data_inicio_calc": {
                    "$cond": {
                        "if": {"$gt": ["$dados_contrato.data_inicio", data_inicial]},
                        "then": "$dados_contrato.data_inicio",
                        "else": data_inicial
                    }
                }
            }
        },
        {
            "$addFields": {
                "dias_corridos_reais": {
                    "$cond": {
                        "if": {
                            "$gte": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_final, "unit": "day"}},
                                0
                            ]
                        },
                        "then": {
                            "$add": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_final, "unit": "day"}},
                                1
                            ]
                        },
                        "else": 0
                    }
                }
            }
        },
        {
            "$addFields": {
                "dias_corridos": "$dias_corridos_reais",
                "proporcao": {
                    "$cond": {
                        "if": {"$gte": ["$dias_corridos_reais", 30]},
                        "then": 1,
                        "else": {"$divide": ["$dias_corridos_reais", 30]}
                    }
                }
            }
        },
        {
            "$addFields": {
                "horas_proporcionais": {
                    "$round": [
                        {"$multiply": ["$proporcao", 30 * 7.3333]},
                        2
                    ]
                }
            }
        },
        {
            "$addFields": {
                "valor_pagamento": {
                    "$round": [
                        {"$cond": {
                            "if": {"$eq": ["$dados_contrato.tipo_remuneracao", "Fixo"]},
                            "then": {"$multiply": ["$dados_contrato.valor_fixo", "$proporcao"]},
                            "else": 0
                        }},
                        2
                    ]
                }
            }
        },
        {
            "$project": {
                "nome": "$informacoes_basicas.nome",
                "cpf": "$informacoes_basicas.cpf",
                "tipo_remuneracao": "$dados_contrato.tipo_remuneracao",
                "valor_fixo": "$dados_contrato.valor_fixo",
                "horas_proporcionais": 1,
                "valor_pagamento": 1
            }
        }
    ]

    folha_pagamento = list(db.colaboradores.aggregate(pipeline))
    return folha_pagamento


def gerar_folha_variavel_sem_garantia(data_inicio: datetime, data_fim: datetime):
    """
    Calcula o valor de pagamento e total de horas apontadas para colaboradores ativos
    com remuneração do tipo 'Variável sem Garantia'.

    Parâmetros:
    - data_inicio (datetime): Data inicial como objeto datetime.
    - data_fim (datetime): Data final como objeto datetime.

    Retorno:
    - list: Lista de dicionários com as informações de cada colaborador.
    """
    db = mongo.db  # Acessa o banco de dados usando a instância global

    pipeline_colaboradores = [
        {"$match": {"ativo": True, "dados_contrato.tipo_remuneracao": "Variavel sem Garantia"}},
        {
            "$addFields": {
                "data_inicio_calc": {
                    "$cond": {
                        "if": {"$gt": ["$dados_contrato.data_inicio", data_inicio]},
                        "then": "$dados_contrato.data_inicio",
                        "else": data_inicio
                    }
                },
                "dias_corridos_reais": {
                    "$cond": {
                        "if": {
                            "$gte": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_fim, "unit": "day"}},
                                0
                            ]
                        },
                        "then": {
                            "$add": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_fim, "unit": "day"}},
                                1
                            ]
                        },
                        "else": 0
                    }
                },
                "proporcao": {
                    "$cond": {
                        "if": {"$gte": ["$dias_corridos_reais", 30]},
                        "then": 1,
                        "else": {"$divide": ["$dias_corridos_reais", 30]}
                    }
                },
                "horas_proporcionais": {
                    "$round": [{"$multiply": ["$proporcao", 30 * 7.3333]}, 2]
                }
            }
        },
        {
            "$project": {
                "_id": 0,
                "cpf": "$informacoes_basicas.cpf",
                "nome": "$informacoes_basicas.nome",
                "tipo_remuneracao": "$dados_contrato.tipo_remuneracao",
                "taxa_hora": {"$toDouble": "$dados_contrato.taxa_hora"},
            }
        }
    ]

    colaboradores = list(db.colaboradores.aggregate(pipeline_colaboradores))
    resultados = []

    for colaborador in colaboradores:
        cpf_filtro = colaborador["cpf"]
        pipeline_horas = [
            {
                "$addFields": {
                    "DT_INICIO_DT": {
                        "$dateFromString": {
                            "dateString": "$DT_INICIO",
                            "format": "%Y-%m-%dT%H:%M:%S.%L%z"
                        }
                    },
                    "MINUTOS_INT": {"$toInt": "$MINUTOS"}
                }
            },
            {
                "$match": {
                    "DT_INICIO_DT": {
                        "$gte": data_inicio,
                        "$lte": data_fim
                    }
                }
            },
            {
                "$lookup": {
                    "from": "usuarios_psoffice",
                    "localField": "USU_ID",
                    "foreignField": "USU_ID",
                    "as": "usuario"
                }
            },
            {"$unwind": "$usuario"},
            {
                "$addFields": {
                    "cpf_numeral": {
                        "$replaceAll": {"input": "$usuario.CPF", "find": ".", "replacement": ""}
                    }
                }
            },
            {
                "$addFields": {
                    "cpf_numeral": {
                        "$replaceAll": {"input": "$cpf_numeral", "find": "-", "replacement": ""}
                    }
                }
            },
            {"$match": {"cpf_numeral": cpf_filtro}},
            {
                "$group": {
                    "_id": "$cpf_numeral",
                    "total_horas": {"$sum": {"$divide": ["$MINUTOS_INT", 60]}}
                }
            },
            {
                "$project": {
                    "_id": 0,
                    "CPF": "$_id",
                    "Total_Horas": {"$round": ["$total_horas", 2]}
                }
            }
        ]

        horas_resultado = list(db.apontamentos_psoffice.aggregate(pipeline_horas))
        total_horas = horas_resultado[0]["Total_Horas"] if horas_resultado else 0
        valor_pagamento = total_horas * colaborador["taxa_hora"]

        resultados.append({
            "cpf": colaborador["cpf"],
            "nome": colaborador["nome"],
            "tipo_remuneracao": colaborador["tipo_remuneracao"],
            "taxa_hora": colaborador["taxa_hora"],
            "total_horas": total_horas,
            "valor_pagamento": round(valor_pagamento, 2),
        })

    return resultados


In [10]:
# Exemplo de uso
data_inicial = datetime(2024, 12, 1)
data_final = datetime(2024, 12, 31)

resultado_folha_fixo = calcular_folha_pagamento_fixo(data_inicial, data_final)
resultado_folha_variavel_sem_garantia = gerar_folha_variavel_sem_garantia(data_inicial, data_final)

resultado_folha = resultado_folha_fixo + resultado_folha_variavel_sem_garantia
resultado_folha


[{'_id': ObjectId('678279e422cd2ca1d2f5cc7f'),
  'horas_proporcionais': 220.0,
  'valor_pagamento': 6500.0,
  'nome': 'Breno Leone da silva Bicalho',
  'cpf': '10637667611',
  'tipo_remuneracao': 'Fixo',
  'valor_fixo': 6500.0},
 {'_id': ObjectId('6782863722cd2ca1d2f5cc80'),
  'horas_proporcionais': 0.0,
  'valor_pagamento': 0.0,
  'nome': 'Iandra Sara',
  'cpf': '00000000000',
  'tipo_remuneracao': 'Fixo',
  'valor_fixo': 4500.0},
 {'_id': ObjectId('678279e422cd2ca1d2f5cc80'),
  'horas_proporcionais': 220.0,
  'valor_pagamento': 5500.0,
  'nome': 'Ana Paula Ferreira de Souza',
  'cpf': '28549785234',
  'tipo_remuneracao': 'Fixo',
  'valor_fixo': 5500.0},
 {'_id': ObjectId('678279e422cd2ca1d2f5cc81'),
  'horas_proporcionais': 220.0,
  'valor_pagamento': 8500.0,
  'nome': 'Carlos Eduardo Menezes',
  'cpf': '12345678901',
  'tipo_remuneracao': 'Fixo',
  'valor_fixo': 8500.0},
 {'cpf': '8790063660',
  'nome': 'Luana Ribeiro Santos',
  'tipo_remuneracao': 'Variavel sem Garantia',
  'taxa_h

In [1]:
from datetime import datetime
from pymongo import MongoClient
import os
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv("/root/projeto-conti-rh/conti-rh-backend/.env")

# Conectar ao MongoDB
client = MongoClient(os.getenv("MONGO_URI"))
db = client.get_database()

def calcular_folha_pagamento_fixo(data_inicial: datetime, data_final: datetime):
    """
    Calcula a folha de pagamento com base na proporção de dias corridos segundo a CLT (mês comercial de 30 dias),
    levando em consideração a data de admissão do colaborador.

    Parâmetros:
    - data_inicial (datetime): Data inicial do período.
    - data_final (datetime): Data final do período.

    Retorno:
    - list: Lista de colaboradores com o valor de pagamento proporcional, dias corridos e horas proporcionais.
    """
    # Padronizar o mês como tendo 30 dias conforme a CLT
    dias_no_mes = 30

    # Pipeline de agregação para calcular a folha de pagamento
    pipeline = [
    {"$match": {"ativo": True, "dados_contrato.tipo_remuneracao": "Fixo"}},
    {
        "$addFields": {
            "data_inicio_calc": {
                "$cond": {
                    "if": {"$gt": ["$dados_contrato.data_inicio", data_inicial]},
                    "then": "$dados_contrato.data_inicio",
                    "else": data_inicial
                }
            }
        }
    },
    {
        "$addFields": {
            "dias_corridos_reais": {
                "$cond": {
                    "if": {"$gte": [{"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_final, "unit": "day"}}, 0]},
                    "then": {"$add": [{"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_final, "unit": "day"}}, 1]},
                    "else": 0
                }
            }
        }
    },
    {
        "$addFields": {
            "dias_corridos": "$dias_corridos_reais",
            "proporcao": {
                "$cond": {
                    "if": {"$gte": ["$dias_corridos_reais", 30]},
                    "then": 1,
                    "else": {"$divide": ["$dias_corridos_reais", 30]}
                }
            }
        }
    },
    {
        "$addFields": {
            "horas_proporcionais": {
                "$round": [
                    {"$multiply": ["$proporcao", 30 * 7.3333]},
                    2
                ]
            }
        }
    },
    {
        "$addFields": {
            # Ajustar o valor do pagamento para duas casas decimais
            "valor_pagamento": {
                "$round": [
                    {"$cond": {
                        "if": {"$eq": ["$dados_contrato.tipo_remuneracao", "Fixo"]},
                        "then": {"$multiply": ["$dados_contrato.valor_fixo", "$proporcao"]},
                        "else": 0
                    }},
                    2
                ]
            }
        }
    },
    {
        "$project": {
            "nome": "$informacoes_basicas.nome",
            "cpf": "$informacoes_basicas.cpf",
            "tipo_remuneracao": "$dados_contrato.tipo_remuneracao",
            "valor_fixo": "$dados_contrato.valor_fixo",
            "horas_proporcionais": 1,
            "valor_pagamento": 1
        }
    }
]

    # Executar a agregação
    folha_pagamento = list(db.colaboradores.aggregate(pipeline))

    return folha_pagamento

def gerar_folha_variavel_sem_garantia(data_inicio: datetime, data_fim: datetime):
    """
    Calcula o valor de pagamento e total de horas apontadas para colaboradores ativos
    com remuneração do tipo 'Variável sem Garantia'.

    Parâmetros:
    - data_inicio (datetime): Data inicial como objeto datetime.
    - data_fim (datetime): Data final como objeto datetime.

    Retorno:
    - list: Lista de dicionários com as informações de cada colaborador.
    """
    # Pipeline inicial para buscar colaboradores
    pipeline_colaboradores = [
        {"$match": {"ativo": True, "dados_contrato.tipo_remuneracao": "Variavel sem Garantia"}},
        {
            "$addFields": {
                "data_inicio_calc": {
                    "$cond": {
                        "if": {"$gt": ["$dados_contrato.data_inicio", data_inicio]},
                        "then": "$dados_contrato.data_inicio",
                        "else": data_inicio
                    }
                },
                "dias_corridos_reais": {
                    "$cond": {
                        "if": {
                            "$gte": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_fim, "unit": "day"}},
                                0
                            ]
                        },
                        "then": {
                            "$add": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_fim, "unit": "day"}}, 1
                            ]
                        },
                        "else": 0
                    }
                },
                "proporcao": {
                    "$cond": {
                        "if": {"$gte": ["$dias_corridos_reais", 30]},
                        "then": 1,
                        "else": {"$divide": ["$dias_corridos_reais", 30]}
                    }
                },
                "horas_proporcionais": {
                    "$round": [{"$multiply": ["$proporcao", 30 * 7.3333]}, 2]
                }
            }
        },
        {
            "$project": {
                "_id": 0,
                "cpf": "$informacoes_basicas.cpf",
                "nome": "$informacoes_basicas.nome",
                "tipo_remuneracao": "$dados_contrato.tipo_remuneracao",
                "taxa_hora": {"$toDouble": "$dados_contrato.taxa_hora"},
            }
        }
    ]
    
    # Executar a pipeline de colaboradores
    colaboradores = list(db.colaboradores.aggregate(pipeline_colaboradores))
    
    resultados = []
    
    # Para cada colaborador, buscar as horas apontadas
    for colaborador in colaboradores:
        cpf_filtro = colaborador["cpf"]
        pipeline_horas = [
            {
                "$addFields": {
                    "DT_INICIO_DT": {
                        "$dateFromString": {
                            "dateString": "$DT_INICIO",
                            "format": "%Y-%m-%dT%H:%M:%S.%L%z"
                        }
                    },
                    "MINUTOS_INT": {"$toInt": "$MINUTOS"}
                }
            },
            {
                "$match": {
                    "DT_INICIO_DT": {
                        "$gte": data_inicio,
                        "$lte": data_fim
                    }
                }
            },
            {
                "$lookup": {
                    "from": "usuarios_psoffice",
                    "localField": "USU_ID",
                    "foreignField": "USU_ID",
                    "as": "usuario"
                }
            },
            {"$unwind": "$usuario"},
            {
                "$addFields": {
                    "cpf_numeral": {
                        "$replaceAll": {"input": "$usuario.CPF", "find": ".", "replacement": ""}
                    }
                }
            },
            {
                "$addFields": {
                    "cpf_numeral": {
                        "$replaceAll": {"input": "$cpf_numeral", "find": "-", "replacement": ""}
                    }
                }
            },
            {"$match": {"cpf_numeral": cpf_filtro}},
            {
                "$group": {
                    "_id": "$cpf_numeral",
                    "total_horas": {"$sum": {"$divide": ["$MINUTOS_INT", 60]}}
                }
            },
            {
                "$project": {
                    "_id": 0,
                    "CPF": "$_id",
                    "Total_Horas": {"$round": ["$total_horas", 2]}
                }
            }
        ]
        
        horas_resultado = list(db.apontamentos_psoffice.aggregate(pipeline_horas))
        total_horas = horas_resultado[0]["Total_Horas"] if horas_resultado else 0
        
        # Calcular o valor do pagamento
        valor_pagamento = total_horas * colaborador["taxa_hora"]
        
        # Combinar as informações
        resultados.append({
            "cpf": colaborador["cpf"],
            "nome": colaborador["nome"],
            "tipo_remuneracao": colaborador["tipo_remuneracao"],
            "taxa_hora": colaborador["taxa_hora"],
            "total_horas": total_horas,
            "valor_pagamento": round(valor_pagamento, 2),
        })
    
    return resultados

# Exemplo de uso
data_inicial = datetime(2024, 12, 1)
data_final = datetime(2024, 12, 31)

resultado_folha_fixo = calcular_folha_pagamento_fixo(data_inicial, data_final)
resultado_folha_variavel_sem_garantia = gerar_folha_variavel_sem_garantia(data_inicial, data_final)



In [2]:
resultado_folha = resultado_folha_fixo + resultado_folha_variavel_sem_garantia
resultado_folha

[{'_id': ObjectId('678279e422cd2ca1d2f5cc7f'),
  'horas_proporcionais': 220.0,
  'valor_pagamento': 6500.0,
  'nome': 'Breno Leone da silva Bicalho',
  'cpf': '10637667611',
  'tipo_remuneracao': 'Fixo',
  'valor_fixo': 6500.0},
 {'_id': ObjectId('6782863722cd2ca1d2f5cc80'),
  'horas_proporcionais': 0.0,
  'valor_pagamento': 0.0,
  'nome': 'Iandra Sara',
  'cpf': '00000000000',
  'tipo_remuneracao': 'Fixo',
  'valor_fixo': 4500.0},
 {'_id': ObjectId('678279e422cd2ca1d2f5cc80'),
  'horas_proporcionais': 220.0,
  'valor_pagamento': 5500.0,
  'nome': 'Ana Paula Ferreira de Souza',
  'cpf': '28549785234',
  'tipo_remuneracao': 'Fixo',
  'valor_fixo': 5500.0},
 {'_id': ObjectId('678279e422cd2ca1d2f5cc81'),
  'horas_proporcionais': 220.0,
  'valor_pagamento': 8500.0,
  'nome': 'Carlos Eduardo Menezes',
  'cpf': '12345678901',
  'tipo_remuneracao': 'Fixo',
  'valor_fixo': 8500.0},
 {'cpf': '8790063660',
  'nome': 'Luana Ribeiro Santos',
  'tipo_remuneracao': 'Variavel sem Garantia',
  'taxa_h

In [110]:
def gerar_folha_variavel_sem_garantia(data_inicio: str, data_fim: str):
    """
    Calcula o valor de pagamento e total de horas apontadas para colaboradores ativos
    com remuneração do tipo 'Variável sem Garantia'.
    
    Parâmetros:
    - data_inicio (str): Data inicial no formato 'YYYY-MM-DD'.
    - data_fim (str): Data final no formato 'YYYY-MM-DD'.
    
    Retorno:
    - list: Lista de dicionários com as informações de cada colaborador.
    """
    # Converter datas para datetime
    data_inicio_dt = datetime.strptime(data_inicio, "%Y-%m-%d")
    data_fim_dt = datetime.strptime(data_fim, "%Y-%m-%d")
    
    # Pipeline inicial para buscar colaboradores
    pipeline_colaboradores = [
        {"$match": {"ativo": True, "dados_contrato.tipo_remuneracao": "Variavel sem Garantia"}},
        {
            "$addFields": {
                "data_inicio_calc": {
                    "$cond": {
                        "if": {"$gt": ["$dados_contrato.data_inicio", data_inicio_dt]},
                        "then": "$dados_contrato.data_inicio",
                        "else": data_inicio_dt
                    }
                },
                "dias_corridos_reais": {
                    "$cond": {
                        "if": {
                            "$gte": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_fim_dt, "unit": "day"}},
                                0
                            ]
                        },
                        "then": {
                            "$add": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_fim_dt, "unit": "day"}}, 1
                            ]
                        },
                        "else": 0
                    }
                },
                "proporcao": {
                    "$cond": {
                        "if": {"$gte": ["$dias_corridos_reais", 30]},
                        "then": 1,
                        "else": {"$divide": ["$dias_corridos_reais", 30]}
                    }
                },
                "horas_proporcionais": {
                    "$round": [{"$multiply": ["$proporcao", 30 * 7.3333]}, 2]
                }
            }
        },
        {
            "$project": {
                "_id": 0,
                "cpf": "$informacoes_basicas.cpf",
                "nome": "$informacoes_basicas.nome",
                "taxa_hora": {"$toDouble": "$dados_contrato.taxa_hora"},
            }
        }
    ]
    
    # Executar a pipeline de colaboradores
    colaboradores = list(db.colaboradores.aggregate(pipeline_colaboradores))
    
    resultados = []
    
    # Para cada colaborador, buscar as horas apontadas
    for colaborador in colaboradores:
        cpf_filtro = colaborador["cpf"]
        pipeline_horas = [
            {
                "$addFields": {
                    "DT_INICIO_DT": {
                        "$dateFromString": {
                            "dateString": "$DT_INICIO",
                            "format": "%Y-%m-%dT%H:%M:%S.%L%z"
                        }
                    },
                    "MINUTOS_INT": {"$toInt": "$MINUTOS"}
                }
            },
            {
                "$match": {
                    "DT_INICIO_DT": {
                        "$gte": data_inicio_dt,
                        "$lte": data_fim_dt
                    }
                }
            },
            {
                "$lookup": {
                    "from": "usuarios_psoffice",
                    "localField": "USU_ID",
                    "foreignField": "USU_ID",
                    "as": "usuario"
                }
            },
            {"$unwind": "$usuario"},
            {
                "$addFields": {
                    "cpf_numeral": {
                        "$replaceAll": {"input": "$usuario.CPF", "find": ".", "replacement": ""}
                    }
                }
            },
            {
                "$addFields": {
                    "cpf_numeral": {
                        "$replaceAll": {"input": "$cpf_numeral", "find": "-", "replacement": ""}
                    }
                }
            },
            {"$match": {"cpf_numeral": cpf_filtro}},
            {
                "$group": {
                    "_id": "$cpf_numeral",
                    "total_horas": {"$sum": {"$divide": ["$MINUTOS_INT", 60]}}
                }
            },
            {
                "$project": {
                    "_id": 0,
                    "CPF": "$_id",
                    "Total_Horas": {"$round": ["$total_horas", 2]}
                }
            }
        ]
        
        horas_resultado = list(db.apontamentos_psoffice.aggregate(pipeline_horas))
        total_horas = horas_resultado[0]["Total_Horas"] if horas_resultado else 0
        
        # Calcular o valor do pagamento
        valor_pagamento = total_horas * colaborador["taxa_hora"]
        
        # Combinar as informações
        resultados.append({
            "CPF": colaborador["cpf"],
            "Nome": colaborador["nome"],
            "Taxa Hora": colaborador["taxa_hora"],
            "Total Horas Apontadas": total_horas,
            "Valor Pagamento": round(valor_pagamento, 2),
        })
    
    return resultados

# Exemplo de uso
data_inicio = "2024-12-1"
data_fim = "2024-12-31"
resultado = gerar_folha_variavel_sem_garantia(data_inicio, data_fim)

# Exibir resultado
for r in resultado:
    print(r)


{'CPF': '8790063660', 'Nome': 'Luana Ribeiro Santos', 'Taxa Hora': 65.0, 'Total Horas Apontadas': 409.0, 'Valor Pagamento': 26585.0}


In [111]:
def gerar_folha_variavel_sem_garantia(data_inicio: datetime, data_fim: datetime):
    """
    Calcula o valor de pagamento e total de horas apontadas para colaboradores ativos
    com remuneração do tipo 'Variável sem Garantia'.

    Parâmetros:
    - data_inicio (datetime): Data inicial como objeto datetime.
    - data_fim (datetime): Data final como objeto datetime.

    Retorno:
    - list: Lista de dicionários com as informações de cada colaborador.
    """
    # Pipeline inicial para buscar colaboradores
    pipeline_colaboradores = [
        {"$match": {"ativo": True, "dados_contrato.tipo_remuneracao": "Variavel sem Garantia"}},
        {
            "$addFields": {
                "data_inicio_calc": {
                    "$cond": {
                        "if": {"$gt": ["$dados_contrato.data_inicio", data_inicio]},
                        "then": "$dados_contrato.data_inicio",
                        "else": data_inicio
                    }
                },
                "dias_corridos_reais": {
                    "$cond": {
                        "if": {
                            "$gte": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_fim, "unit": "day"}},
                                0
                            ]
                        },
                        "then": {
                            "$add": [
                                {"$dateDiff": {"startDate": "$data_inicio_calc", "endDate": data_fim, "unit": "day"}}, 1
                            ]
                        },
                        "else": 0
                    }
                },
                "proporcao": {
                    "$cond": {
                        "if": {"$gte": ["$dias_corridos_reais", 30]},
                        "then": 1,
                        "else": {"$divide": ["$dias_corridos_reais", 30]}
                    }
                },
                "horas_proporcionais": {
                    "$round": [{"$multiply": ["$proporcao", 30 * 7.3333]}, 2]
                }
            }
        },
        {
            "$project": {
                "_id": 0,
                "cpf": "$informacoes_basicas.cpf",
                "nome": "$informacoes_basicas.nome",
                "taxa_hora": {"$toDouble": "$dados_contrato.taxa_hora"},
            }
        }
    ]
    
    # Executar a pipeline de colaboradores
    colaboradores = list(db.colaboradores.aggregate(pipeline_colaboradores))
    
    resultados = []
    
    # Para cada colaborador, buscar as horas apontadas
    for colaborador in colaboradores:
        cpf_filtro = colaborador["cpf"]
        pipeline_horas = [
            {
                "$addFields": {
                    "DT_INICIO_DT": {
                        "$dateFromString": {
                            "dateString": "$DT_INICIO",
                            "format": "%Y-%m-%dT%H:%M:%S.%L%z"
                        }
                    },
                    "MINUTOS_INT": {"$toInt": "$MINUTOS"}
                }
            },
            {
                "$match": {
                    "DT_INICIO_DT": {
                        "$gte": data_inicio,
                        "$lte": data_fim
                    }
                }
            },
            {
                "$lookup": {
                    "from": "usuarios_psoffice",
                    "localField": "USU_ID",
                    "foreignField": "USU_ID",
                    "as": "usuario"
                }
            },
            {"$unwind": "$usuario"},
            {
                "$addFields": {
                    "cpf_numeral": {
                        "$replaceAll": {"input": "$usuario.CPF", "find": ".", "replacement": ""}
                    }
                }
            },
            {
                "$addFields": {
                    "cpf_numeral": {
                        "$replaceAll": {"input": "$cpf_numeral", "find": "-", "replacement": ""}
                    }
                }
            },
            {"$match": {"cpf_numeral": cpf_filtro}},
            {
                "$group": {
                    "_id": "$cpf_numeral",
                    "total_horas": {"$sum": {"$divide": ["$MINUTOS_INT", 60]}}
                }
            },
            {
                "$project": {
                    "_id": 0,
                    "CPF": "$_id",
                    "Total_Horas": {"$round": ["$total_horas", 2]}
                }
            }
        ]
        
        horas_resultado = list(db.apontamentos_psoffice.aggregate(pipeline_horas))
        total_horas = horas_resultado[0]["Total_Horas"] if horas_resultado else 0
        
        # Calcular o valor do pagamento
        valor_pagamento = total_horas * colaborador["taxa_hora"]
        
        # Combinar as informações
        resultados.append({
            "CPF": colaborador["cpf"],
            "Nome": colaborador["nome"],
            "Taxa Hora": colaborador["taxa_hora"],
            "Total Horas Apontadas": total_horas,
            "Valor Pagamento": round(valor_pagamento, 2),
        })
    
    return resultados

# Exemplo de uso
data_inicio = datetime(2024, 12, 1)
data_fim = datetime(2024, 12, 31)
resultado = gerar_folha_variavel_sem_garantia(data_inicio, data_fim)

# Exibir resultado
for r in resultado:
    print(r)


{'CPF': '8790063660', 'Nome': 'Luana Ribeiro Santos', 'Taxa Hora': 65.0, 'Total Horas Apontadas': 409.0, 'Valor Pagamento': 26585.0}


In [103]:
resultado

[{'CPF': '8790063660',
  'Nome': 'Luana Ribeiro Santos',
  'Taxa Hora': 65.0,
  'Total Horas Apontadas': 409.0,
  'Valor Pagamento': 26585.0}]

In [99]:
def listar_apontamentos_por_cpf(cpf_filtro: str):
    """
    Lista todos os apontamentos de um CPF específico, mostrando o ID, USU_ID, a data e horas apontadas.

    Parâmetros:
    - cpf_filtro (str): CPF numeral a ser filtrado (somente números).

    Retorno:
    - list: Lista de dicionários contendo ID, USU_ID, data e horas apontadas.
    """
    pipeline = [
        {
            "$addFields": {
                "DT_INICIO_DT": {
                    "$dateFromString": {
                        "dateString": "$DT_INICIO",
                        "format": "%Y-%m-%dT%H:%M:%S.%L%z"
                    }
                },
                "MINUTOS_INT": {"$toInt": "$MINUTOS"}
            }
        },
        {
            "$lookup": {
                "from": "usuarios_psoffice",
                "localField": "USU_ID",
                "foreignField": "USU_ID",
                "as": "usuario"
            }
        },
        {"$unwind": "$usuario"},
        {
            "$addFields": {
                "cpf_numeral": {
                    "$replaceAll": {"input": "$usuario.CPF", "find": ".", "replacement": ""}
                }
            }
        },
        {
            "$addFields": {
                "cpf_numeral": {
                    "$replaceAll": {"input": "$cpf_numeral", "find": "-", "replacement": ""}
                }
            }
        },
        {"$match": {"cpf_numeral": cpf_filtro}},
        {
            "$project": {
                "_id": 1,
                "USU_ID": 1,
                "Data": "$DT_INICIO_DT",
                "Horas_Apontadas": {"$round": [{"$divide": ["$MINUTOS_INT", 60]}, 2]}
            }
        }
    ]
    
    # Executar a pipeline
    resultado = list(db.apontamentos_psoffice.aggregate(pipeline))
    
    return resultado

# Exemplo de uso
cpf_filtro = "8790063660"  # Substitua pelo CPF que deseja filtrar
resultado = listar_apontamentos_por_cpf(cpf_filtro)

# Exibir resultado
for apontamento in resultado:
    print(f"ID: {apontamento['_id']}, USU_ID: {apontamento['USU_ID']}, Data: {apontamento['Data']}, Horas Apontadas: {apontamento['Horas_Apontadas']}")


ID: 6783262e11f0ad856ed54ace, USU_ID: 25, Data: 2024-12-02 03:00:00, Horas Apontadas: 4.5
ID: 6783262e11f0ad856ed54acf, USU_ID: 25, Data: 2024-12-02 03:00:00, Horas Apontadas: 3.5
ID: 6783262e11f0ad856ed54ad0, USU_ID: 25, Data: 2024-12-03 03:00:00, Horas Apontadas: 4.5
ID: 6783262e11f0ad856ed54ad1, USU_ID: 25, Data: 2024-12-03 03:00:00, Horas Apontadas: 3.5
ID: 6783262e11f0ad856ed54ad2, USU_ID: 25, Data: 2024-12-04 03:00:00, Horas Apontadas: 4.5
ID: 6783262e11f0ad856ed54ad3, USU_ID: 25, Data: 2024-12-04 03:00:00, Horas Apontadas: 3.5
ID: 6783262e11f0ad856ed54ad4, USU_ID: 25, Data: 2024-12-05 03:00:00, Horas Apontadas: 4.5
ID: 6783262e11f0ad856ed54ad5, USU_ID: 25, Data: 2024-12-05 03:00:00, Horas Apontadas: 3.5
ID: 6783262e11f0ad856ed54ad6, USU_ID: 25, Data: 2024-12-06 03:00:00, Horas Apontadas: 4.5
ID: 6783262e11f0ad856ed54ad7, USU_ID: 25, Data: 2024-12-06 03:00:00, Horas Apontadas: 3.5
ID: 6783262e11f0ad856ed54ad8, USU_ID: 25, Data: 2024-12-09 03:00:00, Horas Apontadas: 4.0
ID: 678326